# SGD, Adam et AdamW : comparer une règle de mise à jour

Distinguer le gradient calculé de la règle qui choisit le prochain poids.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#optimiseurs-regularisation-pratique).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


La première boucle utilisait SGD avec un taux fixe. Avant de changer ce choix, observons un seul pas, avec le même poids et la même perte pour chaque optimiseur.

Nous gardons une miniature scalaire : poids=1 et perte=(poids−2)². Le gradient vaut −2. Le taux lr=0,1 règle l’ampleur de la modification.

La décroissance des poids, weight_decay, ajoute ici une contrainte d’amplitude. Pour SGD sans momentum, la direction devient gradient + 0,1×poids = −1,9.

| Objet | Valeur | Rôle |
| --- | --- | --- |
| Poids avant | 1 | Paramètre à modifier |
| Gradient de la perte | −2 | Indication locale fournie par backward |
| Terme de décroissance | 0,1×1 = 0,1 | Contribution supplémentaire |
| Direction totale | −1,9 | Gradient et décroissance combinés |
| Poids après | 1−0,1×(−1,9) = 1,19 | Nouveau point |

Le gradient lu dans poids.grad reste celui de la perte : −2. Le terme de décroissance est appliqué dans la règle de l’optimiseur.

Adam utilise des moyennes du gradient et de son carré. AdamW découple la décroissance des poids de cette adaptation. Un même weight_decay ne signifie donc pas le même calcul.

## Démonstration 1 · Repartir trois fois du même point

Comparer les sorties sans déclarer un optimiseur universellement meilleur.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Répéter la même étape sur les données

La liste contient les trois constructeurs. La boucle crée un poids neuf à chaque passage : les expériences ne se poursuivent pas l’une l’autre.

[poids] est la liste des tenseurs à mettre à jour. lr et weight_decay sont les mêmes valeurs numériques pour cette comparaison locale.

La perte dépend du poids neuf. backward calcule sa dérivée ; item lit −2 pour l’affichage.

step applique la règle choisie. Les résultats distincts viennent des règles d’adaptation et de décroissance, pas d’un gradient de départ différent.

In [4]:
for type_optimiseur in [torch.optim.SGD, torch.optim.Adam, torch.optim.AdamW]:
    poids = torch.tensor(1., requires_grad=True)
    optimiseur = type_optimiseur([poids], lr=0.1, weight_decay=0.1)
    perte = (poids - 2.) ** 2
    perte.backward()
    gradient = poids.grad.item()
    optimiseur.step()
    print(type_optimiseur.__name__, "gradient =", gradient,
          f"nouveau poids = {poids.item():.6f}")

SGD gradient = -2.0 nouveau poids = 1.190000
Adam gradient = -2.0 nouveau poids = 1.100000
AdamW gradient = -2.0 nouveau poids = 1.090000


La comparaison explique une première mise à jour ; elle ne compare pas encore la généralisation sur des sons.

Augmenter weight_decay n’est pas un réglage de volume audio. C’est une décision d’optimisation dont l’intérêt doit se vérifier sur la validation.

### Complément — limiter la norme du gradient

L’écrêtage du gradient, ou gradient clipping, limite sa norme avant step. Il ne remplace ni backward ni le choix d’un taux adapté.

Pour isoler cette opération, nous choisissons perte=10×poids : son gradient vaut 10. Cette perte affine n’a pas de minimum fini ; elle sert seulement à fabriquer ce gradient.

## Démonstration 2 · Lire un gradient avant et après sa limitation

Dans ce cas scalaire, limiter sa norme revient à limiter sa valeur absolue.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [5]:
import torch


### Calculer l’état suivant

Le poids vaut 1. La perte est affine ; backward écrit donc 10 dans grad.

In [6]:
poids = torch.tensor(1., requires_grad=True)
optimiseur = torch.optim.SGD([poids], lr=0.1)
perte = 10 * poids


In [7]:
observer('poids', poids)
observer('perte', perte)

poids forme () type torch.float32 requires_grad True
  valeurs : 1.0
perte forme () type torch.float32 requires_grad True
  valeurs : 10.0


### Calculer les gradients

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [8]:
perte.backward()


In [9]:
observer_entrainement({'poids': poids, 'optimiseur': optimiseur, 'perte': perte}, montrer_poids=False)

poids.grad forme () type torch.float32 requires_grad False
  valeurs : 10.0


### Calculer l’état suivant

clip_grad_norm_ modifie les gradients en place. max_norm=1 fixe la norme maximale ; la fonction retourne la norme précédente, ici 10.

In [10]:
norme_avant = torch.nn.utils.clip_grad_norm_([poids], max_norm=1.)


In [11]:
observer('norme_avant', norme_avant)

norme_avant forme () type torch.float32 requires_grad False
  valeurs : 10.0


### Lire les sorties

SGD lit maintenant ce gradient limité. Avec lr=0,1, le poids devient environ 0,9.

In [12]:
print("norme avant :", norme_avant.item())
print(f"gradient limite : {poids.grad.item():.6f}")


norme avant : 10.0
gradient limite : 1.000000


### Appliquer une mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [13]:
optimiseur.step()


In [14]:
observer_entrainement({'poids': poids, 'optimiseur': optimiseur, 'perte': perte, 'norme_avant': norme_avant}, montrer_poids=True)

poids forme () type torch.float32 requires_grad True
  valeurs : 0.8999999761581421
poids.grad forme () type torch.float32 requires_grad False
  valeurs : 0.9999999403953552


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [15]:
print(f"poids apres : {poids.item():.6f}")

poids apres : 0.900000


L’ordre est backward, puis clipping, puis step. Appliquer l’écrêtage avant backward ne limiterait pas le gradient qui va être calculé.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

AdamW reçoit un poids 1, un gradient nul, lr=0,1 et weight_decay=0,1. Quel poids suit le pas ?

In [16]:
exo_p = torch.nn.Parameter(torch.tensor(1.))
exo_opt = torch.optim.AdamW([exo_p],lr=.1,weight_decay=.1)
exo_p.grad = torch.zeros_like(exo_p)
exo_opt.step()
exo_poids = ...  # Remplacer les points par votre réponse.
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_p.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Avec un gradient nul explicite, la décroissance découplée applique ici le facteur 1−lr×weight_decay.

</details>

### Exercice 2

SGD reçoit un poids 1, un gradient 2, lr=0,1 et weight_decay=0,1. Quel poids suit le pas ?

In [17]:
exo_p = torch.nn.Parameter(torch.tensor(1.))
exo_opt = torch.optim.SGD([exo_p],lr=.1,weight_decay=.1)
exo_p.grad = torch.tensor(2.)
exo_opt.step()
exo_poids = ...  # Remplacer les points par votre réponse.
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_p.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

La contribution de décroissance vaut 0,1×1. Soustrayez 0,1×(2+0,1).

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_p = torch.nn.Parameter(torch.tensor(1.))
exo_opt = torch.optim.AdamW([exo_p],lr=.1,weight_decay=.1)
exo_p.grad = torch.zeros_like(exo_p)
exo_opt.step()
exo_poids = 0.99
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_p.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_p = torch.nn.Parameter(torch.tensor(1.))
exo_opt = torch.optim.SGD([exo_p],lr=.1,weight_decay=.1)
exo_p.grad = torch.tensor(2.)
exo_opt.step()
exo_poids = 0.79
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_p.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [18]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()